# Opportunity Extraction Script

This notebook extracts ticker observations from the `capture_frames` table in the SQLite database, filters for significant price movements (opportunities), and saves them into separate CSV files per symbol.

## Configuration

In [ ]:
import sqlite3
import pandas as pd
import json
import os
from pathlib import Path

# --- Configuration ---
DB_PATH = "/Users/theapemachine/data/symm.sqlite"
MIN_UP_PCT = 3.0  # Minimum percentage increase to be considered an opportunity
OUTPUT_DIR = "extracted_opportunities"
SEQ_STEP = 1  # Set to > 1 for downsampling (e.g., 10)

def ensure_output_dir(directory):
    if os.path.isfile(directory):
        raise ValueError(
            f"Output path {directory} is an existing file, not a directory"
        )
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created directory: {directory}")

ensure_output_dir(OUTPUT_DIR)


## Extraction Logic

This block connects to the database and performs the heavy lifting of JSON parsing and filtering.

In [ ]:
def extract_opportunities(db_path, min_up_pct, seq_step):
    print(f"Connecting to {db_path}...")
    try:
        conn = sqlite3.connect(db_path, timeout=60)
    except sqlite3.OperationalError:
        print("Error: Could not connect to database.")
        return

    sql = f"""
    SELECT
        seq,
        json_extract(CAST(payload AS TEXT), '$.data[0].symbol')     AS symbol,
        CAST(json_extract(CAST(payload AS TEXT), '$.data[0].last')  AS REAL) AS last_price,
        json_extract(CAST(payload AS TEXT), '$.data[0].timestamp')  AS timestamp,
        CAST(json_extract(CAST(payload AS TEXT), '$.data[0].change_pct') AS REAL) AS change_pct,
        CAST(json_extract(CAST(payload AS TEXT), '$.data[0].volume')     AS REAL) AS volume,
        CAST(json_extract(CAST(payload AS TEXT), '$.data[0].high')       AS REAL) AS high,
        CAST(json_extract(CAST(payload AS TEXT), '$.data[0].low')        AS REAL) AS low
    FROM capture_frames
    WHERE json_extract(CAST(payload AS TEXT), '$.channel') = 'ticker'
      AND seq % {seq_step} = 0
    """

    print("Fetching data from database (this may take a while)...")
    df = pd.read_sql_query(sql, conn)
    conn.close()

    if df.empty:
        print("No ticker data found.")
        return

    print(f"Fetched {len(df)} rows. Filtering for opportunities (change_pct >= {min_up_pct}%)...")
    # Filter for opportunities
    df_opportunities = df[df['change_pct'] >= min_up_pct].copy()

    if df_opportunities.empty:
        print("No opportunities found with the current threshold.")
        return

    print(f"Found {len(df_opportunities)} opportunity rows. Saving to CSVs...")
    
    # Group by symbol and save
    for symbol, group in df_opportunities.groupby('symbol'):
        filename = os.path.join(OUTPUT_DIR, f"{symbol.replace('/', '_')}_opportunities.csv")
        group.to_csv(filename, index=False)
        print(f"  Saved {filename}")

    # Save a summary CSV
    df_opportunities.to_csv(os.path.join(OUTPUT_DIR, "all_opportunities_summary.csv"), index=False)
    print("\nDone! Summary saved to all_opportunities_summary.csv")

# Run the extraction
extract_opportunities(DB_PATH, MIN_UP_PCT, SEQ_STEP)
